In [0]:
from pyspark.sql import functions as F


In [0]:
df_clean = spark.read.table("workspace.bronze.employee")
df_clean = (df_clean.withColumnRenamed("createdat", "create_at")
            .withColumnRenamed("updatedat", "update_at")
            .withColumnRenamed("file_name", "file_name")
            .withColumn("create_at",F.date_format("create_at","yyyy-MM-dd")))


In [0]:
df_clean = df_clean.select("empno","ename","dept","salary","create_at","modifiedat")
df_clean.write.format("delta").mode("append").saveAsTable("workspace.silver.employee")

In [0]:
%sql
SELECT * FROM workspace.silver.employee;

In [0]:
from delta.tables import DeltaTable

silver_delta = DeltaTable.forName(spark,"workspace.silver.employee")

silver_delta.alias("target").merge(
    df_clean.alias("source"),
    "target.empno = source.empno"
).whenMatchedUpdate(
    set={
        "ename": "source.ename",
        "dept": "source.dept",
        "salary": "source.salary",
        "create_at": "source.create_at",
        "modifiedat": "source.modifiedat"
    }
).whenNotMatchedInsertAll().execute(
)

In [0]:
# SCD Type 2: keep full history of every employee change
from delta.tables import DeltaTable

target_table = "workspace.silver.employee_scd2"

# Hash the tracked attributes so we can detect actual changes (not just touch-ups)
df_scd2 = (df_clean
    .withColumn("row_hash", F.sha2(F.concat_ws("||", "ename", "dept", "salary"), 256))
)

if not spark.catalog.tableExists(target_table):
    # Initial load: every employee gets its first active version
    (df_scd2
        .withColumn("start_date", F.col("modifiedat"))
        .withColumn("end_date", F.lit("9999-12-31").cast("timestamp"))
        .withColumn("is_current", F.lit(True))
        .write.format("delta").mode("overwrite").saveAsTable(target_table))
else:
    target = DeltaTable.forName(spark, target_table)

    # Step 1: close the current version of employees whose attributes changed
    (target.alias("t").merge(
        df_scd2.alias("s"),
        "t.empno = s.empno AND t.is_current = true"
    )
    .whenMatchedUpdate(
        condition="t.row_hash <> s.row_hash",
        set={
            "end_date": "s.modifiedat",
            "is_current": "false"
        }
    )
    .execute())

    # Step 2: insert new active versions for changed + brand-new employees
    (df_scd2.alias("s")
        .join(spark.table(target_table).filter("is_current = true").alias("t"),
              "empno", "left_anti")
        .withColumn("start_date", F.col("modifiedat"))
        .withColumn("end_date", F.lit("9999-12-31").cast("timestamp"))
        .withColumn("is_current", F.lit(True))
        .write.format("delta").mode("append").saveAsTable(target_table))

# Review the full history
# display(spark.table(target_table).orderBy("empno", "start_date"))
